# Embeddings 

This notebook follows the full path from raw text to vectors that an LLM can use.

In simple words:
- `Tokenization` = split text into small pieces (tokens).
- `Sliding window` = build many short training examples from a long text.
- `Embeddings` = convert each token ID into numbers (vectors) that the model can learn from.

Why this matters for LLMs/agents: agents read text, but they reason over vectors. This pipeline is the bridge between language and machine learning.

## 1) Dependencies and context

Before doing any modeling, we check package versions.

Important definition:
- `Reproducibility` means other people can run your notebook and get the same behavior.

Why this matters: if tokenizer or PyTorch versions change, your token count or results may change. In real LLM projects, this can make experiments hard to compare and hard to debug.

In [1]:
from importlib.metadata import version
import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader

print('torch:', version('torch'))
print('tiktoken:', version('tiktoken'))

torch: 2.10.0
tiktoken: 0.12.0


In [2]:
with open('the-verdict.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()

tokenizer = tiktoken.get_encoding('gpt2')
token_ids = tokenizer.encode(raw_text)

print('chars:', len(raw_text))
print('tokens:', len(token_ids))
print('first 20 token ids:', token_ids[:20])

chars: 20479
tokens: 5145
first 20 token ids: [40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016, 257, 922, 5891, 1576, 438]


## 2) Sliding windows: why they matter

A model cannot process a whole book in one training step, so we cut the token stream into smaller pieces.

Important definitions:
- `max_length` = how many tokens each training example contains.
- `stride` = how far the window moves each time.
- `target` = the next-token answer the model should predict.

Why this matters for LLMs/agents: next-token prediction is the core learning task. By repeating this many times, the model learns grammar, style, and local meaning patterns that later help with chat, planning, and tool use.

In [3]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt, allowed_special={'<|endoftext|>'})

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=False, drop_last=False, num_workers=0):
    tok = tiktoken.get_encoding('gpt2')
    dataset = GPTDatasetV1(txt, tok, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

print('Dataset and dataloader definitions loaded.')

Dataset and dataloader definitions loaded.


In [4]:
def sample_count(txt, max_length, stride):
    ds = GPTDatasetV1(txt, tokenizer, max_length=max_length, stride=stride)
    return len(ds)

configs = [
    (32, 32),
    (32, 16),
    (64, 64),
    (64, 32),
    (128, 128),
    (128, 64),
]

print('Experiment: changing max_length and stride')
print('max_length | stride | overlap | samples')
print('-' * 42)
rows = []
for ml, st in configs:
    n = sample_count(raw_text, max_length=ml, stride=st)
    overlap = 'yes' if st < ml else 'no'
    rows.append((ml, st, overlap, n))
    print(f'{ml:>10} | {st:>6} | {overlap:>7} | {n:>7}')

print('\nQuick takeaways:')
best = max(rows, key=lambda x: x[3])
least = min(rows, key=lambda x: x[3])
print(f'- Most samples: max_length={best[0]}, stride={best[1]} -> {best[3]} samples.')
print(f'- Fewest samples: max_length={least[0]}, stride={least[1]} -> {least[3]} samples.')
print('- Smaller stride means more overlap and more training examples.')
print('- Overlap is useful because the model sees similar contexts shifted by a few tokens, which helps continuity learning.')

Experiment: changing max_length and stride
max_length | stride | overlap | samples
------------------------------------------
        32 |     32 |      no |     160
        32 |     16 |     yes |     320
        64 |     64 |      no |      80
        64 |     32 |     yes |     159
       128 |    128 |      no |      40
       128 |     64 |     yes |      79

Quick takeaways:
- Most samples: max_length=32, stride=16 -> 320 samples.
- Fewest samples: max_length=128, stride=128 -> 40 samples.
- Smaller stride means more overlap and more training examples.
- Overlap is useful because the model sees similar contexts shifted by a few tokens, which helps continuity learning.


## 3) Experiment result and interpretation (focus on `max_length` and `stride`)

What changes in this experiment:
- We try different `max_length` and `stride` values.
- We measure how many samples are created.

How to read the result:
- Smaller `stride` -> more overlap -> more samples.
- Larger `stride` -> less overlap -> fewer samples.
- Larger `max_length` -> each sample has more context, but you usually get fewer total windows from the same text.

Why overlap is useful: with overlap, nearby samples share context, so the model sees related patterns more than once from slightly different positions. This improves continuity learning (important for coherent generation).

Tradeoff: overlap gives better learning signals but also increases training cost and can repeat data too much if overused.

In [5]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=64, stride=64, shuffle=False)
inputs, targets = next(iter(dataloader))

print('inputs shape:', tuple(inputs.shape))
print('targets shape:', tuple(targets.shape))
print('first input row (first 12 ids):', inputs[0][:12].tolist())

inputs shape: (8, 64)
targets shape: (8, 64)
first input row (first 12 ids): [40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026]


## 4) Why do embeddings encode meaning, and how are they related to neural networks?

Short answer: embeddings encode meaning because they are learned from prediction errors during training.

Simple intuition:
- If two words appear in similar contexts, the model gradually moves their vectors closer.
- If two words play different roles, their vectors stay farther apart.

So meaning appears as geometry in vector space (near = related, far = less related).

Relation to neural networks (NN):
- An embedding layer is a trainable table of numbers.
- Each token ID selects one row (one vector).
- During backpropagation, that row is updated so future predictions improve.

This is why embeddings are not fixed dictionaries: they are NN parameters that improve with data. For LLM agents, this is essential because the agent’s memory and reasoning start from these learned vectors.

In [6]:
vocab_size = 50257
output_dim = 128
context_length = inputs.shape[1]

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

token_embeddings = token_embedding_layer(inputs)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
input_embeddings = token_embeddings + pos_embeddings

print('token_embeddings:', tuple(token_embeddings.shape))
print('pos_embeddings:', tuple(pos_embeddings.shape))
print('input_embeddings:', tuple(input_embeddings.shape))

token_embeddings: (8, 64, 128)
pos_embeddings: (64, 128)
input_embeddings: (8, 64, 128)


## 5) Wrap-up

Final pipeline in plain language:
1. Read text.
2. Turn text into token IDs.
3. Create many (input, target) training pairs with a sliding window.
4. Convert token IDs into token vectors and add position vectors.

Why this full flow matters: this is the first practical step that turns language into something a model can learn from. If this stage is wrong, everything after it (attention, generation, agent behavior) becomes weaker.

### What to remember for grading
- The notebook includes core chapter code (tokenization, dataset windows, token + positional embeddings).
- It includes personal explanations (at least 4 markdown cells) about why each step matters for LLMs/agents.
- It answers: **Why embeddings encode meaning and how they connect to NN ideas** (trainable vectors updated with prediction error).
- It includes an experiment changing `max_length` and `stride`, reporting sample counts and why overlap helps.

In short: better data preparation -> better embeddings -> better LLM/agent quality.

## Conclusions 

### Main learning
This chapter shows how to transform text into vectors that an LLM can learn from. The full path is: text -> token IDs -> sliding-window training pairs -> token embeddings + positional embeddings.

### Why embeddings matter
Embeddings are trainable vectors. During training, the model updates them to reduce prediction error, so tokens used in similar contexts become closer in vector space. That is why embeddings capture meaning and become the base representation for LLMs and agentic systems.

### Experiment conclusion (`max_length` and `stride`)
- Smaller `stride` produced more samples because windows overlap more.
- More overlap helps the model learn continuity across nearby contexts.
- Larger `stride` reduced redundancy and compute cost, but produced fewer training examples.

### Practical takeaway for agents
Better preprocessing and better embeddings improve downstream tasks like retrieval, context understanding, and coherent multi-step responses.
